# Feature Engineering — Purchase Propensity Project

In this notebook, I'll use the cleaned dataset from the previous notebook to focus on feature and target windows, and define train and test users.

Setup:

- Feature window: Oct 1-31 (behavioral signal)

- Target window: Nov 1-5 (purchase labels)

- User pool: Nov 1-5 purchasers + Oct-active non-purchasers sampled at 3:1

- Train users: 80% of user pool (used for all modeling work)

- Test users: remaining 20% (locked until final evaluation)

Behavioral data volume varies greatly across categories; I'll discuss focusing on categories with sufficient event volume for a reliable model. Using these categories I'll build a scaffold based on user events and labels on whether user purchased in that category during Nov 1-5.

Contents:
1. Setup
2. Split Windows
3. Define Train / Test Users
4. Build Scaffold
5. Attach Labels
6. Features
7. Save Features

## 1. Setup

In [1]:
from datetime import datetime
import polars as pl

pl.Config.set_tbl_rows(-1)

CLEAN_PARQUET       = '../data/events_clean.parquet'
CUTOFF              = pl.lit('2019-11-01').str.to_datetime('%Y-%m-%d')
TARGET_END          = pl.lit('2019-11-06').str.to_datetime('%Y-%m-%d')
SEED                = 42
TRAIN_FRAC          = 0.8
NON_PURCHASER_RATIO = 3

In [2]:
# Load data
lf = pl.scan_parquet(CLEAN_PARQUET)

lf.select(
    pl.len().alias('rows'),
    pl.col('category_l1').n_unique().alias('unique_l1_categories'),
).collect()

rows,unique_l1_categories
u32,u32
74435542,13


## 2. Split Windows

In [3]:
# Feature window: October behavioral data
oct_lf = lf.filter(pl.col('event_time') < CUTOFF)

# Target window: Nov 1–5 purchases with known category
nov_purchases = (
    lf.filter(
        (pl.col('event_type') == 'purchase') &
        (pl.col('event_time') >= CUTOFF) &
        (pl.col('event_time') < TARGET_END)
    )
    .select(['user_id', 'category_l1'])
    .unique()
    .collect()
)

print(f'Oct events: {oct_lf.select(pl.len()).collect().item():,}')
print(f'Nov 1-5 purchase pairs: {len(nov_purchases):,}')
print(f'Nov 1-5 unique purchasing users: {nov_purchases["user_id"].n_unique():,}')

Oct events: 28,906,113
Nov 1-5 purchase pairs: 56,587
Nov 1-5 unique purchasing users: 54,520


## 3. Define Train / Test Users

I'll includetwo groups in the modeling population:

- **Purchasers**: all users who bought at least once in Nov 1–5, these provide the positive labels
- **Non-purchasers**: Oct-active users who had no purchase in Nov 1–5, sampled at `NON_PURCHASER_RATIO : 1` relative to purchasers

Including non-purchasers makes the scaffold more realistic; the model learns from all active users, not just known buyers. The positive rate drops from ~13% (purchaser-only) to ~3%. I'm not considering additional cold-start users, relying on the model to learn from users with very sparse Oct activity, the natural fallback for no behavioral signal here is `cat_purchase_rate`.

The combined pool is split 80/20 into `train_users` (for all modeling work) and `test_users` (locked away until final evaluation). A validation set will be carved from `train_users` at the start of the modeling notebook.

In [4]:
# Purchasers: all users who bought in Nov 1-5
purchaser_users = nov_purchases.select('user_id').unique()

# Oct-active non-purchasers: had at least one event in Oct, did not buy in Nov 1-5
oct_active_users    = oct_lf.select('user_id').unique().collect()
non_purchaser_users = oct_active_users.join(purchaser_users, on='user_id', how='anti')

# Sample non-purchasers at NON_PURCHASER_RATIO : 1
n_non = min(len(purchaser_users) * NON_PURCHASER_RATIO, len(non_purchaser_users))
# sort before sample: Polars unique/group_by don't guarantee row order, so without
# sorting the seed produces different splits depending on upstream execution order
sampled_non_purchasers = non_purchaser_users.sort('user_id').sample(n=n_non, seed=SEED)

# Combined pool to 80/20 train/test split (test locked until final evaluation)
all_users   = pl.concat([purchaser_users, sampled_non_purchasers]).sort('user_id')
train_users = all_users.sample(fraction=TRAIN_FRAC, seed=SEED)
test_users  = all_users.join(train_users, on='user_id', how='anti')

print(f'Purchasers: {len(purchaser_users):,}')
print(f'Non-purchasers: {len(sampled_non_purchasers):,} ({NON_PURCHASER_RATIO}:1 ratio)')
print(f'Total users: {len(all_users):,}')
print(f'Train users: {len(train_users):,}')
print(f'Test users: {len(test_users):,}')

overlap = train_users.join(test_users, on='user_id', how='inner')
print(f'Overlap (must be 0): {len(overlap)}')

Purchasers: 54,520
Non-purchasers: 163,560 (3:1 ratio)
Total users: 218,080
Train users: 174,464
Test users: 43,616
Overlap (must be 0): 0


## 4. Build Scaffold

- Train scaffold: `train_users` x all categories

- Test scaffold: `test_users` x all categories

In [5]:
# All 13 known L1 categories
categories = (
    lf.select('category_l1')
    .unique()
    .collect()
)
print(f'L1 categories ({len(categories)}):')
print(categories.sort('category_l1'))

L1 categories (13):
shape: (13, 1)
┌──────────────┐
│ category_l1  │
│ ---          │
│ str          │
╞══════════════╡
│ accessories  │
│ apparel      │
│ appliances   │
│ auto         │
│ computers    │
│ construction │
│ country_yard │
│ electronics  │
│ furniture    │
│ kids         │
│ medicine     │
│ sport        │
│ stationery   │
└──────────────┘


In [6]:
def build_scaffold(users: pl.DataFrame, categories: pl.DataFrame) -> pl.DataFrame:
    return users.join(categories, how='cross')

train_scaffold = build_scaffold(train_users, categories)
test_scaffold  = build_scaffold(test_users,  categories)

print(f'Train scaffold: {len(train_scaffold):,} rows ({len(train_users):,} users x {len(categories)} categories)')
print(f'Test scaffold: {len(test_scaffold):,} rows ({len(test_users):,} users x {len(categories)} categories)')

Train scaffold: 2,268,032 rows (174,464 users x 13 categories)
Test scaffold: 567,008 rows (43,616 users x 13 categories)


## 5. Attach Labels

In [7]:
def attach_labels(scaffold: pl.DataFrame, purchases: pl.DataFrame) -> pl.DataFrame:
    return (
        scaffold
        .join(
            purchases.with_columns(pl.lit(1).alias('label')),
            on=['user_id', 'category_l1'],
            how='left',
        )
        .with_columns(pl.col('label').fill_null(0))
    )

train_labeled = attach_labels(train_scaffold, nov_purchases)
test_labeled = attach_labels(test_scaffold,  nov_purchases)

for name, df in [('Train', train_labeled), ('Test', test_labeled)]:
    n_pos  = df['label'].sum()
    n_tot  = len(df)
    rate   = n_pos / n_tot * 100
    print(f'{name}: {n_tot:,} rows | {n_pos:,} positives | positive rate: {rate:.2f}%')

Train: 2,268,032 rows | 45,153 positives | positive rate: 1.99%
Test: 567,008 rows | 11,434 positives | positive rate: 2.02%


In [8]:
# Positive rate per category
(
    train_labeled
    .group_by('category_l1')
    .agg(
        pl.len().alias('scaffold_rows'),
        pl.col('label').sum().alias('positives'),
        (pl.col('label').mean() * 100).round(2).alias('positive_rate_pct'),
    )
    .sort('positive_rate_pct', descending=True)
)

category_l1,scaffold_rows,positives,positive_rate_pct
str,u32,i32,f64
"""electronics""",174464,32333,18.53
"""appliances""",174464,6439,3.69
"""computers""",174464,2384,1.37
"""furniture""",174464,949,0.54
"""apparel""",174464,856,0.49
"""auto""",174464,779,0.45
"""construction""",174464,633,0.36
"""kids""",174464,496,0.28
"""accessories""",174464,157,0.09


Purchase volume per category varies greatly across categories. Some categories don't have enough positive examples to train or evaluate a reliable propensity model. I'll use the natural gap between kids (490) and accessories (165) in the table above as the threshold for including a category in the prediction scaffold.

This filter applies to prediction scaffold only, not the October behavior data. A user who browsed `stationery` in Oct still contributes behavioral signal to their `electronics` propensity; no need to drop that valuable cross-category feature, so `oct_lf` stays untouched.

In [9]:
KEPT_CATS = [
    'electronics', 'appliances', 'computers', 'furniture',
    'apparel', 'auto', 'construction', 'kids'
]

train_labeled = train_labeled.filter(pl.col('category_l1').is_in(KEPT_CATS))
test_labeled  = test_labeled.filter(pl.col('category_l1').is_in(KEPT_CATS))

for name, df in [('Train', train_labeled), ('Test', test_labeled)]:
    n_pos = df['label'].sum()
    n_tot = len(df)
    print(f'{name}: {n_tot:,} rows | {n_pos:,} positives | {n_pos / n_tot * 100:.2f}% positive rate')

Train: 1,395,712 rows | 44,869 positives | 3.21% positive rate
Test: 348,928 rows | 11,365 positives | 3.26% positive rate


## 6. Features

All features are computed from October behavioral data (`oct_lf`) and left-joined onto the scaffold. Five feature groups:

| Group | Features |
|---|---|
| User x category event counts | views, carts, prior purchases x 3 recency windows (7d / 14d / 30d) |
| Recency | days since last view / cart / purchase per (user, category) |
| User-level aggregates | total event volume, session count, active days, category breadth |
| Brand features | unique brands viewed per (user, category) |
| Price features | avg price of views / carts per (user, category) |

The category prior (purchase rate per category) is also a feature, but it is target-derived and partition-dependent, so it is computed in `03_modeling.ipynb` after the val split rather than here.

Null event counts (user had no activity in a category/window) are filled with 0 after ratio features are computed. Null recency values are left as-is, I'll fill these for models that need it in the modeling notebook (GBDT family handle nulls natively).

In [10]:
CUTOFF_DT = datetime(2019, 11, 1) # prediction cutoff
W14_START = datetime(2019, 10, 18) # last 14 days of Oct
W7_START  = datetime(2019, 10, 25) # last 7 days of Oct

### 6.1 User x category event counts

For each (user, category) pair, I'll consider counts of views, carts, and prior purchases across three recency windows:

- the full October window (30d)
- last 14 days (Oct 18–31)
- last 7 days (Oct 25–31)

The three windows together let the model learn both long-term and short-term behavior.

In [11]:
def user_cat_counts(lf: pl.LazyFrame, suffix: str, start=None) -> pl.DataFrame:
    if start is not None:
        lf = lf.filter(pl.col('event_time') >= start)
    return (
        lf
        .group_by(['user_id', 'category_l1'])
        .agg(
            (pl.col('event_type') == 'view').sum().alias(f'views_{suffix}'),
            (pl.col('event_type') == 'cart').sum().alias(f'carts_{suffix}'),
            (pl.col('event_type') == 'purchase').sum().alias(f'purchases_{suffix}'),
        )
        .collect()
    )

counts_30d = user_cat_counts(oct_lf, '30d') # full Oct window
counts_14d = user_cat_counts(oct_lf, '14d', start=W14_START)
counts_7d  = user_cat_counts(oct_lf, '7d',  start=W7_START)

user_cat_events = (
    counts_30d
    .join(counts_14d, on=['user_id', 'category_l1'], how='left')
    .join(counts_7d,  on=['user_id', 'category_l1'], how='left')
)
print(f'User x category event rows: {len(user_cat_events):,}')
user_cat_events.head(3)

User x category event rows: 3,399,639


user_id,category_l1,views_30d,carts_30d,purchases_30d,views_14d,carts_14d,purchases_14d,views_7d,carts_7d,purchases_7d
i32,str,u32,u32,u32,u32,u32,u32,u32,u32,u32
563430664,"""electronics""",3,0,0,3,0,0,null,null,null
519504443,"""electronics""",2,0,0,2,0,0,null,null,null
551090978,"""electronics""",5,0,0,null,null,null,null,null,null


### 6.2 Recency

Days since last view, cart, and purchase per (user, category) relative to the cutoff date (Nov 1). Null means no event of that type in October for that category.

In [12]:
def user_cat_recency(lf: pl.LazyFrame) -> pl.DataFrame:
    return (
        lf
        .group_by(['user_id', 'category_l1'])
        .agg(
            pl.col('event_time').filter(pl.col('event_type') == 'view').max().alias('last_view'),
            pl.col('event_time').filter(pl.col('event_type') == 'cart').max().alias('last_cart'),
            pl.col('event_time').filter(pl.col('event_type') == 'purchase').max().alias('last_purchase'),
        )
        .with_columns(
            (pl.lit(CUTOFF_DT) - pl.col('last_view')).dt.total_days().alias('days_since_view'),
            (pl.lit(CUTOFF_DT) - pl.col('last_cart')).dt.total_days().alias('days_since_cart'),
            (pl.lit(CUTOFF_DT) - pl.col('last_purchase')).dt.total_days().alias('days_since_purchase'),
        )
        .drop(['last_view', 'last_cart', 'last_purchase'])
        .collect()
    )

recency_feats = user_cat_recency(oct_lf)
print(f'Recency feature rows: {len(recency_feats):,}')
recency_feats.head(3)

Recency feature rows: 3,399,639


user_id,category_l1,days_since_view,days_since_cart,days_since_purchase
i32,str,i64,i64,i64
556262014,"""electronics""",23,null,null
564299681,"""appliances""",5,null,null
564934979,"""apparel""",3,null,null


### 6.3 User-level aggregates

Cross-category October behavior: Total event volume by type, session count, active days, and category breadth. These capture overall user engagement across categories.

In [13]:
def user_level_feats(lf: pl.LazyFrame) -> pl.DataFrame:
    return (
        lf
        .group_by('user_id')
        .agg(
            (pl.col('event_type') == 'view').sum().alias('total_views_30d'),
            (pl.col('event_type') == 'cart').sum().alias('total_carts_30d'),
            (pl.col('event_type') == 'purchase').sum().alias('total_purchases_30d'),
            pl.col('event_time').dt.date().n_unique().alias('active_days_30d'),
            pl.col('category_l1').n_unique().alias('category_breadth_30d'),
            pl.col('user_session').n_unique().alias('session_count_30d'),
        )
        .collect()
    )

user_feats = user_level_feats(oct_lf)
print(f'User-level feature rows: {len(user_feats):,}')
user_feats.head(3)

User-level feature rows: 2,408,044


user_id,total_views_30d,total_carts_30d,total_purchases_30d,active_days_30d,category_breadth_30d,session_count_30d
i32,u32,u32,u32,u32,u32,u32
513151545,1,0,0,1,1,1
564334467,9,1,0,2,2,4
544332842,1,0,0,1,1,1


### 6.5 Brand features

Number of unique brands the user viewed in each category during October. Brand is null in ~14% of rows, I'll exclude null-brand events before aggregating, so this feature will itself be null for (user, category)-pairs whose Oct activity was entirely in null-brand rows.

Only the full 30d window is used here since brand breadth is less time-sensitive than event counts.

In [14]:
def user_cat_brand_feats(lf: pl.LazyFrame) -> pl.DataFrame:
    return (
        lf
        .filter(pl.col('brand').is_not_null())
        .group_by(['user_id', 'category_l1'])
        .agg(pl.col('brand').n_unique().alias('brand_count_30d'))
        .collect()
    )

brand_feats = user_cat_brand_feats(oct_lf)
print(brand_feats.shape)
brand_feats.head(3)

(3174868, 3)


user_id,category_l1,brand_count_30d
i32,str,u32
531701962,"""appliances""",1
513421729,"""appliances""",2
552161734,"""electronics""",1


### 6.6 Price features

Average price of items viewed and added to cart per (user, category) in October. Null for (user, category)-pairs with no activity of that event type.

In [15]:
def user_cat_price_feats(lf: pl.LazyFrame) -> pl.DataFrame:
    return (
        lf
        .group_by(['user_id', 'category_l1'])
        .agg(
            pl.col('price').filter(pl.col('event_type') == 'view').mean().alias('avg_price_viewed_30d'),
            pl.col('price').filter(pl.col('event_type') == 'cart').mean().alias('avg_price_carted_30d'),
        )
        .collect()
    )

price_feats = user_cat_price_feats(oct_lf)
print(price_feats.shape)
price_feats.head(3)

(3399639, 4)


user_id,category_l1,avg_price_viewed_30d,avg_price_carted_30d
i32,str,f32,f32
533369346,"""appliances""",138.339996,null
512680473,"""electronics""",463.686554,190.460007
515757896,"""electronics""",165.995239,null


### 6.7 Assemble feature matrix

Here I'll left-join all feature groups onto the scaffold.

For a (user, category)-pair with no Oct activity in a given window, event counts arrive as null from the join. I'll compute ratio features first, and fill event count nulls with 0 afterwards; this way a user with no activity in a (category, window) gets a null entry where a low activity user gets 0.

Recency nulls are left as-is, I'll handle filling these for models that need it in the next notebook.

In [16]:
EVENT_COUNT_COLS = [
    'views_30d', 'carts_30d', 'purchases_30d',
    'views_14d', 'carts_14d', 'purchases_14d',
    'views_7d',  'carts_7d',  'purchases_7d',
]

def assemble_features(labeled: pl.DataFrame) -> pl.DataFrame:
    return (
        labeled
        .join(user_cat_events, on=['user_id', 'category_l1'], how='left')
        .join(recency_feats,   on=['user_id', 'category_l1'], how='left')
        .join(user_feats,      on='user_id',                  how='left')
        .join(brand_feats,     on=['user_id', 'category_l1'], how='left')
        .join(price_feats,     on=['user_id', 'category_l1'], how='left')
        .with_columns(
            # Ratios computed while event counts are still null for zero-activity (user, category) pairs
            (pl.col('carts_30d') / pl.col('views_30d').clip(lower_bound=1)).alias('cart_view_ratio_30d'),
            (pl.col('carts_7d')  / pl.col('views_7d').clip(lower_bound=1)).alias('cart_view_ratio_7d'),
        )
        .with_columns(
            # Fill event count nulls with 0: no activity in that (category, window) means 0 events
            [pl.col(c).fill_null(0) for c in EVENT_COUNT_COLS]
        )
    )

train_features = assemble_features(train_labeled)
test_features  = assemble_features(test_labeled)

feature_cols = [c for c in train_features.columns if c not in ['user_id', 'category_l1', 'label']]
print(f'Train features: {train_features.shape}')
print(f'Test features:  {test_features.shape}')
print(f'Feature count:  {len(feature_cols)}')
print(f'Columns: {feature_cols}')
train_features.head(3)

Train features: (1395712, 26)
Test features:  (348928, 26)
Feature count:  23
Columns: ['views_30d', 'carts_30d', 'purchases_30d', 'views_14d', 'carts_14d', 'purchases_14d', 'views_7d', 'carts_7d', 'purchases_7d', 'days_since_view', 'days_since_cart', 'days_since_purchase', 'total_views_30d', 'total_carts_30d', 'total_purchases_30d', 'active_days_30d', 'category_breadth_30d', 'session_count_30d', 'brand_count_30d', 'avg_price_viewed_30d', 'avg_price_carted_30d', 'cart_view_ratio_30d', 'cart_view_ratio_7d']


user_id,category_l1,label,views_30d,carts_30d,purchases_30d,views_14d,carts_14d,purchases_14d,views_7d,carts_7d,purchases_7d,days_since_view,days_since_cart,days_since_purchase,total_views_30d,total_carts_30d,total_purchases_30d,active_days_30d,category_breadth_30d,session_count_30d,brand_count_30d,avg_price_viewed_30d,avg_price_carted_30d,cart_view_ratio_30d,cart_view_ratio_7d
i32,str,i32,u32,u32,u32,u32,u32,u32,u32,u32,u32,i64,i64,i64,u32,u32,u32,u32,u32,u32,u32,f32,f32,f64,f64
560609083,"""construction""",0,0,0,0,0,0,0,0,0,0,null,null,null,42,1,0,1,4,2,null,null,null,null,null
560609083,"""kids""",0,0,0,0,0,0,0,0,0,0,null,null,null,42,1,0,1,4,2,null,null,null,null,null
560609083,"""computers""",0,0,0,0,0,0,0,0,0,0,null,null,null,42,1,0,1,4,2,null,null,null,null,null


## 7. Save Features

In [17]:
FEATURES_DIR = '../data/'
train_features.write_parquet(FEATURES_DIR + 'train_features.parquet')
test_features.write_parquet(FEATURES_DIR + 'test_features.parquet')
print(f'Saved: train_features.parquet ({train_features.shape[0]:,} rows, {train_features.shape[1]} cols)')
print(f'Saved: test_features.parquet ({test_features.shape[0]:,} rows, {test_features.shape[1]} cols)')

Saved: train_features.parquet (1,395,712 rows, 26 cols)
Saved: test_features.parquet (348,928 rows, 26 cols)
